# RunPod Serverless, From Zero to a Production API

The notebook behind the demo: sign up → API key → **this notebook** → pay-per-second GPU inference.

What we prove, in order:
1. An idle endpoint keeps **zero workers** — nothing to pay when nothing runs
2. The **first request from zero** (cold start: queue + worker spin-up)
3. A **warm request** — the same call once a worker is active
4. A **burst** of concurrent requests — the queue absorbing load
5. **The bill** — what this whole session actually cost

## 1. Setup

Config lives in `.env` (`RUNPOD_API_KEY`, `ENDPOINT_ID`, `GPU_HOURLY_USD`).
The endpoint can be one you deployed yourself (any Hub template) or a Runpod-hosted public model endpoint — the API is identical.

If you need to create your own endpoint first:
`uv run scripts/create_endpoint.py --template-id <HUB_TEMPLATE_ID>`

In [11]:
import json
import os
import time

import httpx
from dotenv import load_dotenv

load_dotenv('../.env')  # adjust to '../.env' if run from notebooks/

API_KEY = os.environ['RUNPOD_API_KEY']
ENDPOINT_ID = os.environ['ENDPOINT_ID']
GPU_HOURLY_USD = float(os.getenv('GPU_HOURLY_USD', '0.69'))

BASE_URL = f'https://api.runpod.ai/v2/{ENDPOINT_ID}'
#BASE_URL = 'https://api.runpod.ai/v2/txyl1agrms7eiz'
HEADERS = {'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'}
PROMPT = '''Create a photorealistic editorial fashion portrait of the same woman seated low inside an old cream-colored drawing room, surrounded by a dense pack of Dalmatians. The image should feel calm but tense, as if the room belongs to the dogs and she is the still center of their movement.

Place the camera low and close, slightly below her seated eye line, with several Dalmatians pressing into the foreground as soft partial obstructions. One dog crosses the lower edge of the frame out of focus, another rests its head across her lap, and two more flank her shoulders in the midground, creating a spotted circular frame around her face. Keep her face unobstructed and emotionally central.

Pose her seated on a low ivory sofa with one knee angled toward the camera and the other leg folded beneath the skirt, creating a diagonal line through the body. Her torso turns slightly away while her face returns toward the lens. One hand rests calmly on the neck of the Dalmatian across her lap, while the other gathers the skirt near her hip. Her expression is quiet, guarded, and watchful rather than sweet.

Dress her in a structured warm-ivory mohair knit dress with a fitted waist, long narrow sleeves, and a softly flared skirt with real ribbing, seam logic, and believable fabric weight. The dress should not look like a loose nightgown or generic gauze. Add an off-white leather belt with a small oxidized silver buckle to define the waist and interrupt the softness. Jewelry should be deliberate: sculptural oxidized silver earrings, a heavy silver cuff, and one black onyx ring. No delicate chain filler.

The room should have aged plaster walls, an ivory wool sofa, pale stone flooring, a dark antique wood side table partly visible behind the dogs, and heavy cream curtains filtering overcast daylight. The environment stays restrained and pale, but not blank. Let the Dalmatian spots become the strongest graphic pattern in the image.

Lighting should be soft window light from one side, with subtle shadow falloff across her cheekbones, collarbone, sleeves, and the dogs’ bodies. Add a faint cool gray bounce from the stone floor and mild background falloff so the pack recedes naturally behind her.

Palette hierarchy: hero pattern black-and-white Dalmatian spots, support color warm ivory, restraint color oxidized silver, neutral stabilizers cream plaster, pale stone, antique dark wood, and natural skin warmth.

The final image should read as a quiet high-fashion portrait compressed by a living spotted pack: elegant, strange, controlled, and slightly dangerous. Maintain realistic skin texture, believable dog anatomy, natural fabric tension, grounded body weight, clean face priority, and editorial depth. Avoid pet-photo sweetness, blank studio softness, costume styling, fantasy princess mood, and generic lifestyle portraiture.'


'''

print(f'Endpoint: {BASE_URL}')

Endpoint: https://api.runpod.ai/v2/black-forest-labs-flux-1-schnell


## 2. Health check — how many workers are running right now?

Min workers = 0 means the endpoint **scales to zero** when idle — this check is where we confirm there's nothing running and nothing to pay for.

(Public model endpoints don't expose `/health`; your own endpoints do.)

In [9]:
with httpx.Client(timeout=30.0) as client:
    resp = client.get(f'{BASE_URL}/health', headers=HEADERS)

if resp.status_code == 401:
    print('/health not available on this endpoint (normal for public model endpoints).')
else:
    resp.raise_for_status()
    print(json.dumps(resp.json(), indent=2))

/health not available on this endpoint (normal for public model endpoints).


## 3. Cold start — the first request from zero workers

Submit an async job (`POST /run`), then poll `GET /status/{id}`. The first call pays the cold start: worker provisioning + model load. `delayTime` in the response is exactly that cost, in milliseconds.

In [12]:
with httpx.Client() as client:
    t0 = time.monotonic()
    resp = client.post(
        f'{BASE_URL}/run', headers=HEADERS,
        json={'input': {'prompt': PROMPT}}, timeout=30.0,
    )
    resp.raise_for_status()
    job_id = resp.json()['id']
    print(f'job submitted: {job_id}')

    last = None
    while True:
        job = client.get(f'{BASE_URL}/status/{job_id}', headers=HEADERS, timeout=30.0).json()
        if job['status'] != last:
            print(f"t={time.monotonic() - t0:6.1f}s  status -> {job['status']}")
            last = job['status']
        if job['status'] == 'COMPLETED':
            cold = job
            break
        assert job['status'] not in ('FAILED', 'CANCELLED'), job
        time.sleep(2)

print(f"cold start delayTime: {cold['delayTime']} ms, executionTime: {cold['executionTime']} ms")

from IPython.display import Image as IPyImage, display

if isinstance(cold.get('output'), dict) and isinstance(cold['output'].get('result'), str):
    display(IPyImage(url=cold['output']['result']))
else:
    print('Output:', json.dumps(cold['output'])[:300])

job submitted: 2c6fdc3f-e3bc-4f11-ac7c-d861fa202b6f-e2
t=   0.5s  status -> IN_QUEUE
t=  68.5s  status -> IN_PROGRESS
t=  81.8s  status -> COMPLETED
cold start delayTime: 67581 ms, executionTime: 12745 ms


## 4. Warm request — the FlashBoot difference

Same call, immediately after, via `/runsync` (synchronous — waits for the result). With a worker already active and FlashBoot enabled, `delayTime` should collapse from seconds to milliseconds.

In [14]:
with httpx.Client() as client:
    t0 = time.monotonic()
    resp = client.post(
        f'{BASE_URL}/runsync?wait=120000', headers=HEADERS,
        json={'input': {'prompt': PROMPT}}, timeout=150.0,
    )
    resp.raise_for_status()
    warm = resp.json()
    warm_wall = time.monotonic() - t0

print(f"warm delayTime:    {warm['delayTime']:>6} ms")
print(f"cold delayTime:    {cold['delayTime']:>6} ms")
print(f"difference:        {cold['delayTime'] - warm['delayTime']:>6} ms faster when warm")
print()
print('Output:', json.dumps(warm['output'])[:200])
print()
if isinstance(warm.get('output'), dict) and isinstance(warm['output'].get('result'), str):
    display(IPyImage(url=warm['output']['result']))

warm delayTime:     16919 ms
cold delayTime:     67581 ms
difference:         50662 ms faster when warm

Output: {"cost": 0.003, "result": "https://image.runpod.ai/wavespeed-flux-schnell/7cc9874fd1bb4b9a97af18fa0e9ee0e1/result.jpeg"}



## 5. Burst — 20 concurrent requests

Fire 20 requests at once and watch how the platform absorbs them: each lands in the queue, workers pick jobs up as they scale, and every request still completes.

In [15]:
import asyncio

TOPICS = [
    'GPU virtualization', 'quantization of LLMs', 'KV caching', 'speculative decoding',
    'LoRA adapters', 'mixture-of-experts', 'flash attention', 'gradient checkpointing',
    'RAG pipelines', 'vector databases', 'RLHF', 'distillation',
    'batch inference', 'CUDA streams', 'tensor parallelism', 'pipeline parallelism',
    'activation checkpointing', 'model sharding', 'token healing', 'continuous batching',
]

async def one(client, prompt):
    t = time.monotonic()
    r = await client.post(
        f'{BASE_URL}/runsync?wait=300000', headers=HEADERS,
        json={'input': {'prompt': prompt}}, timeout=320.0,
    )
    r.raise_for_status()
    job = r.json()
    job['_wall_s'] = time.monotonic() - t
    return job

t0 = time.monotonic()
async with httpx.AsyncClient() as client:
    results = await asyncio.gather(*(one(client, f'An icon of {t}') for t in TOPICS))
burst_wall = time.monotonic() - t0

completed = [j for j in results if j['status'] == 'COMPLETED']
walls = sorted(j['_wall_s'] for j in completed)
delays = sorted(j['delayTime'] for j in completed)
print(f'completed:            {len(completed)} / {len(results)}')
print(f'burst wall time:      {burst_wall:.1f}s')
print(f'request wall median:  {walls[len(walls)//2]:.1f}s   max: {walls[-1]:.1f}s')
print(f'delayTime median:     {delays[len(delays)//2]} ms')
print()

urls = [
    j['output']['result'] for j in results
    if isinstance(j.get('output'), dict) and isinstance(j['output'].get('result'), str)
]
if urls:
    from IPython.display import HTML, display

    rows = [
        '<tr>' + ''.join(
            f"<td style='padding:3px'><img src='{u}' width='170'/></td>"
            for u in urls[i:i + 5]
        ) + '</tr>'
        for i in range(0, len(urls), 5)
    ]
    display(HTML('<table style="border-collapse:collapse">' + ''.join(rows) + '</table>'))
    print(f'{len(urls)} generated images, one per request.')

completed:            20 / 20
burst wall time:      83.7s
request wall median:  56.5s   max: 83.7s
delayTime median:     47408 ms



,,,,
,,,,
,,,,
,,,,


20 generated images, one per request.


## 6. The bill

Per-second billing, from worker start to full stop — **queue waiting is not billed**, so the estimate uses `executionTime` (active GPU work), not `delayTime`. Some endpoints report the exact billed cost per job in the output; when they don't, we estimate from execution time × the GPU's hourly rate.

In [16]:
sections = {
    'cold start': [cold],
    'warm request': [warm],
    'burst (20 requests)': list(results),
}

def reported_cost(jobs):
    costs = [
        j['output']['cost'] for j in jobs
        if isinstance(j.get('output'), dict) and j['output'].get('cost') is not None
    ]
    return sum(costs) if costs else None

def estimate_cost(jobs):
    exec_s = sum(j.get('executionTime') or 0 for j in jobs) / 1000.0
    return exec_s, exec_s * GPU_HOURLY_USD / 3600.0

total_reported, total_est = 0.0, 0.0
for name, jobs in sections.items():
    rep = reported_cost(jobs)
    exec_s, est = estimate_cost(jobs)
    if rep is not None:
        print(f'{name:22s} billed: ${rep:.5f}')
        total_reported += rep
    else:
        print(f'{name:22s} ~{exec_s:.1f}s GPU time -> ${est:.5f} (est. at ${GPU_HOURLY_USD}/hr)')
        total_est += est

print()
if total_reported:
    print(f'Total billed this session: ${total_reported:.5f}')
else:
    print(f'Total estimated worker cost: ${total_est:.5f}')
print()
print('That is the entire economics of the demo: a real GPU, real concurrency, pennies.')

cold start             billed: $0.00300
warm request           billed: $0.00300
burst (20 requests)    billed: $0.06000

Total billed this session: $0.06600

That is the entire economics of the demo: a real GPU, real concurrency, pennies.


## Wrap-up

**Runpod is the AI Developer Cloud. Sign Up Today.** — link in the video description.

Reproduce this end to end:
1. Sign up and create an API key (console → Settings → API Keys — use a *Restricted* key)
2. Deploy an endpoint: console → Serverless → New Endpoint, or `scripts/create_endpoint.py`
3. Put the key and endpoint ID in `.env`
4. Run this notebook top to bottom